# Implementing Explainable AI Techniques (SHAP, LIME)

## 📚 Learning Objectives

By completing this notebook, you will:
- Implement SHAP for model explanation
- Implement LIME for local explanations
- Explain model predictions
- Interpret feature importance
- Apply explainability to AI models

## 🔗 Where this fits

**Builds on:** Course 01 (AIAT 111) — Unit 3, lesson 05 "Model Interpretability: SHAP and LIME" — the same two libraries, consolidated here into one comparison you can defend to somebody who was refused.

**Used later in:** Course 08 (AIAT 122) — Unit 4, lesson 04 "Ethical Concerns: Bias, Fairness, Interpretability".

---

This notebook covers practical activities from **Course 06, Unit 4**:
- Implementing explainable AI techniques (SHAP, LIME)

---

## Introduction

**Explainable AI (XAI)** techniques like SHAP and LIME provide insights into model decisions, enabling transparency and accountability in AI systems.


## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- The **real Titanic passenger manifest** (`Course 04/datasets/raw/titanic.csv`) and the
  same six-feature survival model built in Notebooks 01-02, so the SHAP and LIME
  results here are directly comparable with those notebooks.
- `shap`, `lime`, `scikit-learn`.

**Outputs:** What you'll see when you run the cells

- SHAP global importance and a SHAP local explanation for one real passenger.
- A LIME explanation for that same passenger.
- A side-by-side comparison: where the two methods agree, and where they disagree.

---


In [1]:
# Concept map: the two workhorse XAI methods before using them side by side below.
# Why first: SHAP and LIME answer the same question with different guarantees -
# knowing the difference tells you which to reach for.

import shap
import lime
import numpy as np

print("✅ Libraries imported!")
print("\nExplainable AI Techniques")
print("=" * 60)

# SHAP divides a prediction fairly among features (game theory) -
# consistent and additive, at a higher computational price.
print("\nSHAP (SHapley Additive exPlanations):")
print(" - Game theory-based")
print(" - Feature importance")
print(" - Global and local explanations")
print(" - Consistent explanations")

# LIME fits a tiny readable model around one prediction -
# fast and intuitive, but results vary between runs.
print("\nLIME (Local Interpretable Model-agnostic Explanations):")
print(" - Local explanations")
print(" - Model-agnostic")
print(" - Perturbation-based")
print(" - Interpretable models")

print("\nApplications:")
print(" - Model debugging")
print(" - Feature importance")
print(" - Regulatory compliance")
print(" - User trust")

print("\n✅ Explainable AI concepts understood!")

✅ Libraries imported!

Explainable AI Techniques

SHAP (SHapley Additive exPlanations):
 - Game theory-based
 - Feature importance
 - Global and local explanations
 - Consistent explanations

LIME (Local Interpretable Model-agnostic Explanations):
 - Local explanations
 - Model-agnostic
 - Perturbation-based
 - Interpretable models

Applications:
 - Model debugging
 - Feature importance
 - Regulatory compliance
 - User trust

✅ Explainable AI concepts understood!


In [2]:
# WHY side by side: running BOTH methods on the same model and the same real
# person shows where they agree (a signal you can trust) and where they differ
# (an artifact of the method, not a fact about the passenger).

# Practice: compute BOTH SHAP and LIME explanations on one model
import numpy as np
import pandas as pd
import shap
from lime.lime_tabular import LimeTabularExplainer
import warnings
warnings.filterwarnings('ignore')  # silence sklearn feature-name warning from LIME's numpy inputs
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Same REAL setup as Notebooks 01-02: the Titanic manifest, six interpretable
# features, one recorded outcome. Explaining a model trained on real people is
# what makes the explanation a statement about the world.
df = pd.read_csv('../../../Course 04/datasets/raw/titanic.csv')
df['Age'] = df['Age'].fillna(df['Age'].median())      # 177 ages are genuinely missing
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])  # 2 ports missing

X = pd.DataFrame({
    'Pclass':    df['Pclass'].astype(float),      # ticket class: 1 = first, 3 = third
    'Age':       df['Age'].astype(float),         # years (median-imputed where missing)
    'SibSp':     df['SibSp'].astype(float),       # siblings/spouse aboard
    'Parch':     df['Parch'].astype(float),       # parents/children aboard
    'Fare':      df['Fare'].astype(float),        # ticket price paid
    'is_female': (df['Sex'] == 'female').astype(float),  # 1 = female, 0 = male
})
y = df['Survived'].values                          # 1 = survived, 0 = died

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25,
                                                    random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=150, random_state=42).fit(X_train, y_train)
print(f"Real passengers: {len(df)} (train {len(X_train)} / test {len(X_test)})")
print(f"Model test accuracy: {accuracy_score(y_test, model.predict(X_test)):.3f}")

# ---- SHAP: exact, additive attributions for the whole test set ----
sv = shap.TreeExplainer(model).shap_values(X_test)
sv1 = sv[1] if isinstance(sv, list) else (sv[:, :, 1] if sv.ndim == 3 else sv)
shap_rank = pd.Series(np.abs(sv1).mean(axis=0), index=X_test.columns
                      ).sort_values(ascending=False)
print("\nSHAP global importance (mean |SHAP|):")
for f, v in shap_rank.items():
    print(f"  {f:<15} {v:.4f}")

# ---- LIME: a fast local surrogate around ONE real passenger ----
i = 0
proba = model.predict_proba(X_test.iloc[[i]])[0, 1]
lime_exp = LimeTabularExplainer(
    X_train.values, feature_names=list(X_train.columns),
    class_names=['died', 'survived'], mode='classification', random_state=42,
).explain_instance(X_test.iloc[i].values, model.predict_proba, num_features=6)
print(f"\nPassenger #{i} of the test set (a real person from the manifest):")
print(X_test.iloc[i].round(2).to_string())
print(f"Model says: survival probability {proba:.2f}; "
      f"this passenger actually {'survived' if y_test[i] == 1 else 'died'}.")
print("\nLIME explanation (local surrogate rules):")
for rule, w in lime_exp.as_list():
    print(f"  {rule:<35} {w:+.4f}")

# ---- SHAP local, same passenger, for a like-for-like comparison ----
shap_local = pd.Series(sv1[i], index=X_test.columns)
print("\nSHAP local attributions for the SAME passenger:")
for f, v in shap_local.sort_values(key=abs, ascending=False).items():
    print(f"  {f:<15} {v:+.4f}")

# ---- Do the two methods agree? Compare their top local feature, computed ----
shap_top = shap_local.abs().idxmax()
lime_top = max(lime_exp.as_map()[1], key=lambda t: abs(t[1]))[0]
lime_top_name = X_test.columns[lime_top]
print("\n" + "="*70)
print(f"SHAP's strongest factor for this passenger: {shap_top}")
print(f"LIME's strongest factor for this passenger: {lime_top_name}")
print("They AGREE on the dominant factor." if shap_top == lime_top_name
      else "They DISAGREE on the dominant factor.")
print(f"Globally, SHAP ranks '{shap_rank.index[0]}' as the most influential feature")
print(f"across all {len(X_test)} test passengers ({shap_rank.iloc[0]:.4f} mean |SHAP|).")

# Agreement is not all-or-nothing: compare the FULL local ordering, not just #1.
shap_order = list(shap_local.abs().sort_values(ascending=False).index)
lime_order = [X_test.columns[j] for j, _ in
              sorted(lime_exp.as_map()[1], key=lambda t: -abs(t[1]))]
matches = sum(a == b for a, b in zip(shap_order, lime_order))
print(f"\nSHAP order: {shap_order}")
print(f"LIME order: {lime_order}")
print(f"Positions where the two orderings match: {matches} of {len(shap_order)}")
print("\nThe headline factor is the same, and the tail ordering is not. That is the")
print("practical lesson: SHAP is exact and additive for this tree model, LIME fits")
print("a fast approximate surrogate around one point, so their minor ranks drift.")
print("Report which method produced an explanation - an explanation without its")
print("method is not evidence, and a rank you cannot reproduce is not a finding.")


Real passengers: 891 (train 668 / test 223)
Model test accuracy: 0.789



SHAP global importance (mean |SHAP|):


  is_female       0.2040
  Fare            0.0910
  Pclass          0.0865
  Age             0.0634
  SibSp           0.0292
  Parch           0.0201

Passenger #0 of the test set (a real person from the manifest):
Pclass        3.00
Age          30.00
SibSp         0.00
Parch         0.00
Fare          8.05
is_female     0.00
Model says: survival probability 0.08; this passenger actually died.

LIME explanation (local surrogate rules):
  is_female <= 0.00                   -0.3775
  2.00 < Pclass <= 3.00               -0.0637
  7.90 < Fare <= 13.86                -0.0499
  SibSp <= 0.00                       +0.0474
  Parch <= 0.00                       -0.0252
  28.00 < Age <= 36.00                -0.0176

SHAP local attributions for the SAME passenger:
  is_female       -0.1193
  Fare            -0.1071
  Age             -0.0418
  Pclass          -0.0256
  Parch           -0.0150
  SibSp           +0.0058

SHAP's strongest factor for this passenger: is_female
LIME's strongest facto

## 📚 References

1. Ribeiro, M. T., Singh, S. & Guestrin, C. (2016). *"Why Should I Trust You?": Explaining the Predictions of Any Classifier*. KDD 2016. <https://arxiv.org/abs/1602.04938>
2. Lundberg, S. M. & Lee, S.-I. (2017). *A Unified Approach to Interpreting Model Predictions*. NeurIPS 2017. <https://arxiv.org/abs/1705.07874>
3. Barredo Arrieta, A., Díaz-Rodríguez, N., Del Ser, J., et al. (2020). *Explainable Artificial Intelligence (XAI): Concepts, Taxonomies, Opportunities and Challenges toward Responsible AI*. Information Fusion, 58. <https://arxiv.org/abs/1910.10045>
4. Slack, D., Hilgard, S., Jia, E., Singh, S. & Lakkaraju, H. (2020). *Fooling LIME and SHAP: Adversarial Attacks on Post hoc Explanation Methods*. AIES 2020. <https://arxiv.org/abs/1911.02508>